# Preprocessing
This preprocessing script converts the raw traffic accident dataset into two processed datasets:

- `data/processed/traffic_accidents_cleaned.csv`
- `data/processed/traffic_accidents_model_ready.csv`

The cleaned dataset keeps human-readable values for analysis.
The model-ready dataset encodes categorical variables so that it can be used directly for machine learning.


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, LabelEncoder



# Paths
RAW_PATH = Path("data/raw/traffic_accidents.csv")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# 1. Load raw data
df_raw = pd.read_csv(RAW_PATH)

# 2. Define columns
feature_cols = [
    "traffic_control_device",
    "weather_condition",
    "lighting_condition",
    "first_crash_type",
    "trafficway_type",
    "alignment",
    "roadway_surface_cond",
    "road_defect",
    "crash_type",
    "intersection_related_i",
    "prim_contributory_cause",
    "num_units",
]

target_col = "most_severe_injury"

required_cols = feature_cols + [target_col]

missing_cols = [col for col in required_cols if col not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in raw dataset: {missing_cols}")



# 3. Create working dataset
df = df_raw[required_cols].copy()

# Add stable row identifier for traceability.
df.insert(0, "record_id", np.arange(1, len(df) + 1))


# 4. Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

feature_cols = [col.lower() for col in feature_cols]
target_col = target_col.lower()

cat_cols = [col for col in feature_cols if col != "num_units"]
num_cols = ["num_units"]



# 5. Standardize categorical values
for col in cat_cols + [target_col]:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Treat common placeholder strings as missing.
    df[col] = df[col].replace({
        "": pd.NA,
        "NAN": pd.NA,
        "NULL": pd.NA,
        "NONE": pd.NA,
        "UNKNOWN": pd.NA,
        "UNKNOWN/NA": pd.NA,
        "NOT APPLICABLE": pd.NA,
    })



# 6. Convert numeric columns
df["num_units"] = pd.to_numeric(df["num_units"], errors="coerce")



# 7. Remove exact duplicate records
# Since this dataset does not contain a unique crash ID,
# duplicates are checked using all raw columns instead of selected modeling columns.
raw_duplicate_mask = df_raw.duplicated(keep="first")
rows_removed_exact_duplicates = int(raw_duplicate_mask.sum())

df = df.loc[~raw_duplicate_mask].copy()


# 8. Handle missing values
# Target is required for supervised learning.
df = df.dropna(subset=[target_col]).copy()

# Categorical predictors:
# use explicit MISSING category instead of dropping rows.
for col in cat_cols:
    df[col] = df[col].fillna("MISSING")

# Numeric predictor:
# median imputation + missingness flag.
df["num_units_was_missing"] = df["num_units"].isna().astype(int)
num_units_median = df["num_units"].median()
df["num_units"] = df["num_units"].fillna(num_units_median)



# 9. Add validation / anomaly flags
# Flag them for later analysis.
df["flag_num_units_nonpositive"] = (df["num_units"] <= 0).astype(int)

q1 = df["num_units"].quantile(0.25)
q3 = df["num_units"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

df["flag_num_units_high_outlier"] = (df["num_units"] > upper_bound).astype(int)



# 10. Group rare categorical levels

# This reduces extremely sparse one-hot columns.
rare_threshold = 0.01

for col in cat_cols:
    freq = df[col].value_counts(normalize=True, dropna=False)
    rare_categories = freq[freq < rare_threshold].index
    df[col] = df[col].where(~df[col].isin(rare_categories), "OTHER_RARE")



# 11. Save cleaned, human-readable dataset

cleaned_cols = (
    ["record_id"]
    + cat_cols
    + [
        "num_units",
        "num_units_was_missing",
        "flag_num_units_nonpositive",
        "flag_num_units_high_outlier",
    ]
    + [target_col]
)

df_cleaned = df[cleaned_cols].copy()

df_cleaned.to_csv(
    PROCESSED_DIR / "traffic_accidents_cleaned.csv",
    index=False
)


 
# 12. Build model-ready encoded dataset
 
# One-hot encoding is used for nominal categorical predictors
# to avoid creating artificial order.
try:
    onehot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    # Compatibility with older scikit-learn versions.
    onehot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

X_cat = onehot_encoder.fit_transform(df_cleaned[cat_cols])
onehot_feature_names = onehot_encoder.get_feature_names_out(cat_cols)

df_cat_encoded = pd.DataFrame(
    X_cat,
    columns=onehot_feature_names,
    index=df_cleaned.index
)

numeric_and_flag_cols = [
    "num_units",
    "num_units_was_missing",
    "flag_num_units_nonpositive",
    "flag_num_units_high_outlier",
]

label_encoder = LabelEncoder()
target_encoded = label_encoder.fit_transform(df_cleaned[target_col])

df_model_ready = pd.concat(
    [
        df_cleaned[["record_id"]],
        df_cat_encoded,
        df_cleaned[numeric_and_flag_cols],
        pd.Series(
            target_encoded,
            name="most_severe_injury_encoded",
            index=df_cleaned.index
        ),
    ],
    axis=1
)

df_model_ready.to_csv(
    PROCESSED_DIR / "traffic_accidents_model_ready.csv",
    index=False
)


 
# 13. Console output
 
print("Done.")
print(f"Cleaned dataset saved to: {PROCESSED_DIR / 'traffic_accidents_cleaned.csv'}")
print(f"Model-ready dataset saved to: {PROCESSED_DIR / 'traffic_accidents_model_ready.csv'}")
print(f"Cleaned shape: {df_cleaned.shape}")
print(f"Model-ready shape: {df_model_ready.shape}")

Done.
Cleaned dataset saved to: data\processed\traffic_accidents_cleaned.csv
Model-ready dataset saved to: data\processed\traffic_accidents_model_ready.csv
Cleaned shape: (209275, 17)
Model-ready shape: (209275, 70)


# Summary

In [11]:
print("\n[1] Input and Output")
print(f"Raw input file: {RAW_PATH}")
print(f"Cleaned output file: {PROCESSED_DIR / 'traffic_accidents_cleaned.csv'}")
print(f"Model-ready output file: {PROCESSED_DIR / 'traffic_accidents_model_ready.csv'}")

print("\n[2] Dataset Shape")
print(f"Raw dataset shape: {df_raw.shape}")
print(f"After preprocessing cleaned dataset shape: {df_cleaned.shape}")
print(f"Model-ready dataset shape: {df_model_ready.shape}")

print("\n[3] Selected Columns")
print(f"Number of selected predictor columns: {len(feature_cols)}")
print("Predictor columns:")
for col in feature_cols:
    print(f"  - {col}")
print(f"Target column: {target_col}")

print("\n[4] Column Types")
print("Categorical predictor columns:")
for col in cat_cols:
    print(f"  - {col}")
print("Numeric predictor columns:")
for col in num_cols:
    print(f"  - {col}")

print("\n[5] Duplicate Removal Summary")
print(f"Exact duplicate raw records removed: {rows_removed_exact_duplicates}")

print("\n[6] Missing Values in Cleaned Dataset")
missing_counts = df_cleaned.isna().sum()
missing_rates = df_cleaned.isna().mean()

for col in df_cleaned.columns:
    print(
        f"  - {col}: "
        f"{missing_counts[col]} missing "
        f"({missing_rates[col]:.4%})"
    )

print("\n[7] Target Class Balance")
class_counts = df_cleaned[target_col].value_counts(dropna=False)
class_rates = df_cleaned[target_col].value_counts(normalize=True, dropna=False)

for cls in class_counts.index:
    print(
        f"  - {cls}: "
        f"{class_counts[cls]} rows "
        f"({class_rates[cls]:.4%})"
    )

print("\n[8] Numeric Feature Summary")
print(df_cleaned["num_units"].describe())

print("\n[9] Outlier Treatment Summary")
print("Outlier treatment method for num_units: IQR rule")
print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"High outlier upper bound: {upper_bound}")
print(f"Rows where num_units was originally missing: {df_cleaned['num_units_was_missing'].sum()}")
print(f"Rows with non-positive num_units flag: {df_cleaned['flag_num_units_nonpositive'].sum()}")
print(f"Rows with high num_units outlier flag: {df_cleaned['flag_num_units_high_outlier'].sum()}")
print("Outliers were flagged, not removed.")

print("\n[10] Categorical Cardinality After Rare Category Grouping")
for col in cat_cols:
    print(f"  - {col}: {df_cleaned[col].nunique()} unique values")

print("\n[11] Rare Category Grouping Check")
for col in cat_cols:
    other_rare_count = (df_cleaned[col] == "OTHER_RARE").sum()
    print(f"  - {col}: {other_rare_count} rows grouped as OTHER_RARE")

print("\n[12] Encoding Summary")
print("Categorical predictors were encoded using OneHotEncoder.")
print(f"Number of one-hot encoded feature columns: {len(onehot_feature_names)}")
print("Target variable was encoded using LabelEncoder.")

print("\nTarget encoding mapping:")
for original_class, encoded_value in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  - {original_class} -> {encoded_value}")

print("\n[13] Cleaned Dataset Schema")
for col, dtype in df_cleaned.dtypes.items():
    print(f"  - {col}: {dtype}")

print("\n[14] Model-Ready Dataset Schema Summary")
print(f"Total columns: {df_model_ready.shape[1]}")
print("Column groups:")
print("  - record_id")
print("  - one-hot encoded categorical predictor columns")
print("  - numeric and flag columns")
print("  - most_severe_injury_encoded")
print("Model-ready dataset dtype counts:")
print(df_model_ready.dtypes.value_counts())

print("\n[15] Final Output Confirmation")
print(f"traffic_accidents_cleaned.csv rows: {df_cleaned.shape[0]}")
print(f"traffic_accidents_cleaned.csv columns: {df_cleaned.shape[1]}")
print(f"traffic_accidents_model_ready.csv rows: {df_model_ready.shape[0]}")
print(f"traffic_accidents_model_ready.csv columns: {df_model_ready.shape[1]}")

print("\nData quality summary information printed successfully.")
print("=" * 80)


[1] Input and Output
Raw input file: data\raw\traffic_accidents.csv
Cleaned output file: data\processed\traffic_accidents_cleaned.csv
Model-ready output file: data\processed\traffic_accidents_model_ready.csv

[2] Dataset Shape
Raw dataset shape: (209306, 24)
After preprocessing cleaned dataset shape: (209275, 17)
Model-ready dataset shape: (209275, 70)

[3] Selected Columns
Number of selected predictor columns: 12
Predictor columns:
  - traffic_control_device
  - weather_condition
  - lighting_condition
  - first_crash_type
  - trafficway_type
  - alignment
  - roadway_surface_cond
  - road_defect
  - crash_type
  - intersection_related_i
  - prim_contributory_cause
  - num_units
Target column: most_severe_injury

[4] Column Types
Categorical predictor columns:
  - traffic_control_device
  - weather_condition
  - lighting_condition
  - first_crash_type
  - trafficway_type
  - alignment
  - roadway_surface_cond
  - road_defect
  - crash_type
  - intersection_related_i
  - prim_contribu